## Python notebook for generating the STAC catalog json and corresponding Item json for Raster & Vector layers

### Tools:
1. Pystac 
2. Rasterio
3. Geopandas
4. Matplotlib

This notebook returns Catalog json for Raster and Vector layers.

### 1. Importing the required modules

In [1]:
import os
import json
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

import requests
from io import BytesIO

import rasterio
from rasterio.warp import transform_bounds
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
import pystac
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import constants
import numpy as np
from shapely.geometry import mapping, box
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


### 2. Defining the variables used in the notebook

In [2]:
base_dir="../data/"

blocks_info = [
    #     {
    #     "block": "gobindpur",
    #     "district":"saraikela-kharsawan",
    #     "state": "jharkhand",
    #     "lulc_raster_file": "raster_files/saraikela-kharsawan_gobindpur_2023-07-01_2024-06-30_LULCmap_10m.tif",
    #     "surface_water_bodies_file": "swb2_saraikela-kharsawan_gobindpur.geojson",
    #     "lulc_raster_style_file": "style_file.qml",
    #     "surface_water_bodies_style_file": "swb_style.qml"
    # },
    # {
    #     "block": "mirzapur",
    #     "district":"mirzapur",
    #     "state": "uttar_pradesh",
    #     "lulc_raster_file":"raster_files/Mirzapur_Mirzapur_2023-07-01_2024-06-30_LULCmap_10m.tif",
    #     "surface_water_bodies_file":"surface_waterbodies_mirzapur_mirzapur.geojson",
    #     "lulc_raster_style_file": "style_file.qml",
    #     "surface_water_bodies_style_file": "swb_style.qml"
    # },
    # {
    #     "block": "koraput",
    #     "district":"koraput",
    #     "state": "odisha",
    #     "lulc_raster_file": "raster_files/Narayanpatana_Koraput_2023-07-01_2024-06-30_LULCmap_10m.tif",
    #     "surface_water_bodies_file": "surface_waterbodies_koraput_narayanpatana.geojson",
    #     "lulc_raster_style_file": "style_file.qml",
    #     "surface_water_bodies_style_file": "swb_style.qml"
    # },
    {
        "block": "badlapur",
        "district":"jaunpur",
        "state": "uttar_pradesh",
        "admin_boundary_file":"admin_boundary_jaunpur_badlapur.geojson",
        # "nrega_assets_file":"input/vector_files/jaunpur_badlapur.geojson",
        "lulc_raster_file":"jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif",
        # "terrain_raster_file":"input/raster_files/terrain_raster_jaunpur_badlapur.tif",
        # "terrain_vector_file":"input/vector_files/jaunpur_badlapur_terrain_clusters.geojson",
        # "clart_file":"input/raster_files/clart_jaunpur_badlapur.tif",
        # "surface_water_bodies_file":"swb3_jaunpur_badlapur.geojson",
        # "drainage_lines_file":"drainage_lines_jaunpur_badlapur.geojson",
        # "change_detection_raster_file":"change_jaunpur_badlapur_Afforestation.tif",
        # "cropping_intensity_file":"cropping_intensity_jaunpur_badlapur_2017-23.geojson",
        # "tree_health_ccd_raster_file":"tree_health_ccd_raster_jaunpur_badlapur_2022.tif",
        # "Prec_annual_file":"Prec_annual_jaunpur_badlapur.geojson",
        # "tree_health_ch_raster_file":"tree_health_ch_raster_jaunpur_badlapur_2021.tif",
        # "tree_health_overall_raster_file":"tree_health_overall_change_raster_jaunpur_badlapur.tif",
        # "aquifer_vector_file":"aquifer_vector_jaunpur_badlapur.geojson",
        # "soge_vector_file":"soge_vector_jaunpur_badlapur.geojson",
        # "restoration_file":"restoration_jaunpur_badlapur_raster.tif",
        # "change_detection_raster_CropIntensity_file":"change_jaunpur_badlapur_CropIntensity.tif",
        # "change_detection_raster_Deforestation_file":"change_jaunpur_badlapur_Deforestation.tif",
        # "change_detection_raster_Degradation_file":"change_jaunpur_badlapur_Degradation.tif",
        # "change_detection_raster_Urbanization_file":"change_jaunpur_badlapur_Urbanization.tif",
        # "drought_frequency_file":"drought_jaunpur_badlapur_2017_2022.geojson",
        # "runoff_annual_file":"Runoff_annual_jaunpur_badlapur.geojson",
        # "well_depth_annual_file":"well_depth_net_value_jaunpur_badlapur.geojson",
        # "deltaG_annual_file":"filtered_delta_g_annual_jaunpur_badlapur_uid.geojson",
        # "deltaG_fortnight_file":"filtered_delta_g_fortnight_jaunpur_badlapur_uid.geojson",

        "admin_boundary_style_file":"Administrative-Boundary-Style.qml",
        # "nrega_assets_style_file":"input/style_files/swb_style.qml",
        "lulc_raster_style_file":"LULC0_12class.qml",
        # "terrain_raster_style_file":"input/style_files/terrain_1-12class.qml",
        # "terrain_vector_style_file":"input/style_files/Terrain-Vector-Layer-Style.qml",
        # "clart_style_file":"input/style_files/CLART-Layer-Style.qml",
        # "surface_water_bodies_style_file":"Surface-Waterbody-style.qml",
        # "drainage_lines_style_file":"Drainage-Layer-Style.qml",
        # "change_detection_raster_style_file":"Afforestation_climate_change.qml",
        # "cropping_intensity_style_file":"Cropping_intensity.qml",
        # "tree_health_ccd_raster_style_file":"ccd_style.qml",
        # "Prec_annual_style_file":"Precipitation_Style.qml",
        # "tree_health_ch_raster_style_file":"Tree_health_style_oac.qml",
        # "tree_health_overall_raster_style_file":"Tree_health_style_oac.qml",
        # "aquifer_vector_style_file":"Aquifer_style.qml",
        # "soge_vector_style_file":"SOGE_style.qml",
        # "restoration_style_file":"Restoration_style.qml",
        # "change_detection_raster_CropIntensity_style_file":"Cropping_Intensity_climate_change.qml",
        # "change_detection_raster_Deforestation_style_file":"Deforestation_climate_change.qml",
        # "change_detection_raster_Degradation_style_file":"Degradation_climate_change.qml",
        # "change_detection_raster_Urbanization_style_file":"Urbanization_climate_change.qml",
        # "drought_frequency_style_file":"Drought_style.qml",
        # "runoff_annual_style_file":"Runoff_style.qml",
        # "well_depth_annual_style_file":"MWS-Well-Depth-18_23.qml",
        # "deltaG_annual_style_file":"MWS-Well-Depth-18_23.qml",
        # "deltaG_fortnight_style_file":"MWS-Well-Depth-18_23.qml"
    }
]

corestack_dir = os.path.join(base_dir, 'CorestackCatalogs')

In [3]:
corestack_dir

'../data/CorestackCatalogs'

In [4]:
blocks_info

[{'block': 'badlapur',
  'district': 'jaunpur',
  'state': 'uttar_pradesh',
  'admin_boundary_file': 'admin_boundary_jaunpur_badlapur.geojson',
  'lulc_raster_file': 'jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif',
  'admin_boundary_style_file': 'Administrative-Boundary-Style.qml',
  'lulc_raster_style_file': 'LULC0_12class.qml'}]

### 3. For Raster layers the data range fecthed from filename

In [5]:
def extract_raster_dates_from_filename(raster_filename):
    try:
        print(raster_filename)
        parts = raster_filename.split('_')
        start_date = datetime.strptime(parts[2], "%Y-%m-%d")
        end_date = datetime.strptime(parts[3], "%Y-%m-%d")
        print(start_date)
        print(end_date)
    except Exception as e:
        raise ValueError(f"Failed to extract raster dates from filename '{raster_filename}': {e}")
        
    return start_date, end_date    

### 4. Parsing the QML file for Raster Layers

In [6]:
def parse_qml_classes_from_url(qml_url):
    try:
        response = requests.get(qml_url, verify=False)
        response.raise_for_status()
        
    
        tree = ET.parse(BytesIO(response.content))
        root = tree.getroot()
        classes = []

        for entry in root.findall(".//paletteEntry"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)

        if not classes:
            for entry in root.findall(".//item"):
                class_info = {}
                for attr_key, attr_value in entry.attrib.items():
                    if attr_key == "value":
                        try:
                            class_info[attr_key] = int(attr_value)
                        except ValueError:
                            class_info[attr_key] = attr_value
                    else:
                        class_info[attr_key] = attr_value
                classes.append(class_info)
        return classes

    except requests.exceptions.RequestException as e:
        print(f"Error fetching QML file from URL: {e}")
        return None
    except ET.ParseError as e:
        print(f"Error parsing XML from QML file: {e}")
        return None

### 5. Generating the vector thumbnails from the qml files 

In [7]:
def rgba_to_hex(rgba_tuple):
    if rgba_tuple is None:
        return '#808080'  # Default gray
    r, g, b, a = rgba_tuple
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

def extract_styling_info(symbol_element):
    fill_color = None
    outline_color = None
    line_width = None

    if symbol_element is None:
        return fill_color, outline_color, line_width

    
    fill_layer = symbol_element.find('.//layer[@class="SimpleFill"]')
    if fill_layer is not None:
        color_option = fill_layer.find('Option[@name="color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                fill_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                fill_color = None

        outline_option = fill_layer.find('Option[@name="outline_color"]')
        if outline_option is not None:
            try:
                rgb_parts = [int(p) for p in outline_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None
        
        width_option = fill_layer.find('Option[@name="outline_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None

    
    line_layer = symbol_element.find('.//layer[@class="SimpleLine"]')
    if line_layer is not None:
        color_option = line_layer.find('Option[@name="line_color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None
        
        width_option = line_layer.find('Option[@name="line_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None
    
    return fill_color, outline_color, line_width


def parse_qml_style(qml_url):
    
    try:
        response = requests.get(qml_url, verify=False)
        response.raise_for_status()       
        tree = ET.parse(BytesIO(response.content))

        root = tree.getroot()
        renderer_element = root.find('.//renderer-v2')

        if renderer_element is None:
            print("No renderer-v2 element found.")
            return None

        renderer_type = renderer_element.get('type')
        style = {'renderer_type': renderer_type}
        symbols = {s.get('name'): s for s in root.findall('.//symbols/symbol')}

        if renderer_type == 'singleSymbol':
            symbol_element = renderer_element.find('.//symbol') or symbols.get(renderer_element.get('symbol'))
            if symbol_element is not None:
                
                color_option = symbol_element.find('.//layer/Option[@name="line_color"]') or symbol_element.find('.//layer/Option[@name="color"]')
                if color_option is not None:
                    color_value = color_option.get('value').split(',')[0:3]
                    rgb_parts = [int(p) for p in color_value]
                    style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                else:
                    
                    color_prop = symbol_element.find('.//prop[@k="color"]')
                    if color_prop is not None:
                        rgb_parts = [int(p) for p in color_prop.get('v').split(',')[:3]]
                        style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                    else:
                        print(f"Warning: Single symbol color not found in {qml_path}.")
                        return None
            else:
                print(f"Warning: Could not find symbol element for singleSymbol in {qml_path}.")
                return None

        elif renderer_type == 'categorizedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['categories'] = []
            for cat in renderer_element.findall('categories/category'):
                symbol_element = cat.find('symbol') or symbols.get(cat.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['categories'].append({
                    'value': cat.get('value'),
                    'label': cat.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        elif renderer_type == 'graduatedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['classes'] = []
            for cls in renderer_element.findall('classes/class'):
                symbol_element = cls.find('symbol') or symbols.get(cls.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['classes'].append({
                    'lower_bound': float(cls.get('lower')),
                    'upper_bound': float(cls.get('upper')),
                    'label': cls.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })
        
        elif renderer_type == 'RuleRenderer':
            style['rules'] = []
            for rule in renderer_element.findall('.//rule'):
                symbol_element = rule.find('.//symbol')
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['rules'].append({
                    'filter': rule.get('filter'),
                    'label': rule.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        else:
            print(f"Warning: Unsupported renderer type '{renderer_type}'. Using default style.")
            return None
        return style
    except Exception as e:
        print(f"Error parsing QML file {qml_url}: {e}")
        return None


In [8]:
def generate_vector_thumbnail(vector_path, out_path, qml_path):
    try:
        response = requests.get(vector_path)
        response.raise_for_status()
        gdf = gpd.read_file(response.text)
        style_info = parse_qml_style(qml_path)

        fig, ax = plt.subplots(figsize=(6, 6))
        
        default_fill_color = (0.8, 0.8, 0.8, 1.0) # Light gray
        default_outline_color = (0, 0, 0, 1.0)   # Black
        default_line_width = 1.0

        if style_info is None:
            print("Applying default style due to parsing error.")
            gdf.plot(ax=ax, color=rgba_to_hex(default_fill_color), edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)
        
        elif style_info.get('renderer_type') == 'singleSymbol':
            print("Applying single symbol style...")
            fill_color = style_info.get('fill_color', default_fill_color)
            outline_color = style_info.get('outline_color', default_outline_color)
            line_width = style_info.get('line_width', default_line_width)
            gdf.plot(ax=ax, color=rgba_to_hex(fill_color), edgecolor=rgba_to_hex(outline_color), linewidth=line_width)

        elif style_info.get('renderer_type') == 'categorizedSymbol':
            print("Applying categorized style...")
            
            color_map = {
                cat.get('value'): rgba_to_hex(cat.get('fill_color', default_fill_color))
                for cat in style_info.get('categories', [])
            }
            
            outline_color_map = {
                cat.get('value'): rgba_to_hex(cat.get('outline_color', default_outline_color))
                for cat in style_info.get('categories', [])
            }

            attribute_name = style_info.get('attribute')

            if attribute_name not in gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                gdf.plot(ax=ax, color=rgba_to_hex(default_fill_color), edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)
            else:
                gdf['mapped_value'] = gdf[attribute_name].apply(lambda x: str(x).strip() if pd.notnull(x) else None)
                
                fill_colors = gdf['mapped_value'].map(color_map)
                fill_colors = fill_colors.fillna(rgba_to_hex(default_fill_color))

                outline_colors = gdf['mapped_value'].map(outline_color_map)
                outline_colors = outline_colors.fillna(rgba_to_hex(default_outline_color))
                
                gdf.plot(ax=ax, color=fill_colors, edgecolor=outline_colors, linewidth=default_line_width)
            

        elif style_info.get('renderer_type') == 'graduatedSymbol':
            print("Applying graduated style...")
            attribute_name = style_info.get('attribute')
            if attribute_name not in gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                gdf.plot(ax=ax, color=rgba_to_hex(default_fill_color), edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)
            else:
                fill_colors = []
                for _, row in gdf.iterrows():
                    val = row[attribute_name]
                    found_color = default_fill_color
                    for cls in style_info.get('classes', []):
                        if cls.get('lower_bound') is not None and cls.get('upper_bound') is not None:
                            if cls['lower_bound'] <= val < cls['upper_bound']:
                                found_color = cls.get('fill_color', default_fill_color)
                                break
                    fill_colors.append(rgba_to_hex(found_color))
                
                gdf.plot(ax=ax, color=fill_colors, edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)

        elif style_info.get('renderer_type') == 'RuleRenderer':
            print("Applying rule-based style...")
            fill_colors = []
            for _, row in gdf.iterrows():
                assigned_color = default_fill_color
                for rule in style_info.get('rules', []):
                    try:
                        attribute_name = rule['filter'].split(' ')[0].strip().strip('"').strip("'")
                        if attribute_name in row and pd.eval(rule['filter'], local_dict={attribute_name: row[attribute_name]}):
                            assigned_color = rule.get('fill_color', default_fill_color)
                            break
                    except Exception:
                        continue 
                fill_colors.append(rgba_to_hex(assigned_color))
            gdf.plot(ax=ax, color=fill_colors, edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)
            

        else:
            print("Applying default blue style.")
            gdf.plot(ax=ax, color='lightblue', edgecolor=rgba_to_hex(default_outline_color), linewidth=default_line_width)

        ax.set_axis_off()
        plt.tight_layout()
        plt.savefig(out_path)
        plt.close(fig)
        print(f"Thumbnail saved to: {out_path}")

    except Exception as e:
        print(f"Error generating vector thumbnail: {e}")

#### 6. Generating the raster thumbnails from the qml files 

In [9]:
def generate_raster_thumbnail(tif_path, out_path, qml_path):
    coverage_id = "LULC_level_1:LULC_23_24_badlapur_level_1"
    params = {
        "service": "WCS",
        "version": "2.0.1",
        "request": "GetCoverage",
        "CoverageId": coverage_id,
        #"bbox": bbox,
        "format": "geotiff"
    }
    print(qml_path)
    response = requests.get(tif_path, params=params, verify=False)
    response.raise_for_status()
    raster_data = BytesIO(response.content)
    with rasterio.open(raster_data) as src:
        arr = src.read(1) 
        nodata = src.nodata
        if nodata is not None:
            arr = np.ma.masked_equal(arr, nodata)
    
    unique_raster_values = np.unique(arr.compressed() if isinstance(arr, np.ma.MaskedArray) else arr)
    print(f"Unique values in raster data: {unique_raster_values}")
    
    style_info = parse_qml_classes_from_url(qml_path)

    # Filter QML info to only include values present in the raster data
    filtered_style_info = [cls for cls in style_info if cls.get('value') in unique_raster_values]
    
    values = [cls['value'] for cls in filtered_style_info if 'value' in cls]
    colors = [cls['color'] for cls in filtered_style_info if 'color' in cls]
    
    print(f"Parsed QML values: {values}")
    print(f"Parsed QML colors: {colors}")
    
    
    try:
        if not values or not colors or len(values) != len(colors):
            raise ValueError("Invalid or insufficient palette information in QML file.")
    
        sorted_indices = np.argsort(values)
        sorted_values = np.array(values)[sorted_indices]
        sorted_colors = np.array(colors)[sorted_indices]

        cmap = ListedColormap(sorted_colors)
        bounds = np.array(sorted_values) - 0.5
        bounds = np.append(bounds, sorted_values[-1] + 0.5)
        norm = Normalize(vmin=bounds.min(), vmax=bounds.max())

    except ValueError as e:
        print(f"Skipping palette generation due to error: {e}. Using a default colormap.")
        cmap = 'gray'
        norm = None

    plt.figure(figsize=(3, 3), dpi=100)
    
    plt.imshow(arr, cmap=cmap, norm=norm, interpolation='none')
    plt.axis('off')

    #os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


### 7. Creating the Raster items and adding the assets

In [10]:
def create_raster_item(state,district, block, raster_filename, raster_path, block_catalog_dir, raster_thumbnail, raster_style_file, base_dir, data_url, title, stac_output_dir, thumbnail_path):
    
    is_terrain = "terrain" in raster_filename.lower()

    if is_terrain:
        start_date = constants.SRTM_DEM_START_DATE
        end_date = constants.SRTM_DEM_END_DATE
        print(f"Detected terrain raster file. Using default dates: {start_date} to {end_date}")
    else:
        try:
            start_date, end_date = extract_raster_dates_from_filename(raster_filename=raster_filename)
        except ValueError as e:
            print(f"Warning: Could not extract dates from filename {raster_filename}. Using default dates.")
            start_date = constants.DEFAULT_START_DATE
            end_date = constants.DEFAULT_END_DATE
            
    coverage_id = "LULC_level_1:LULC_23_24_badlapur_level_1"

    layer_desc_df = pd.read_csv('../data/layer descriptions - desc.csv')
    layer_desc_df = layer_desc_df[layer_desc_df['layer_name'] == 'lulc_raster']
    layer_description = layer_desc_df['layer_description'].iloc[0]

    params = {
        "service": "WCS",
        "version": "2.0.1",
        "request": "GetCoverage",
        "CoverageId": coverage_id,
        #"bbox": bbox,
        "format": "geotiff"
    }

    response = requests.get(raster_path, params=params, verify=False)
    response.raise_for_status()
    raster_data = BytesIO(response.content)

    with rasterio.open(raster_data) as src:
        bounds = src.bounds
        geom = mapping(box(*bounds))
        data_type = str(src.dtypes[0])
        height, width = src.shape
        data_crs = src.crs

        proj_epsg = None
        if src.crs and src.crs.is_epsg_code:
            proj_epsg = src.crs.to_epsg()
        
        if proj_epsg != 32644:
            reprojected_bounds = transform_bounds(src.crs, 'EPSG:32644', *bounds)
            bbox = list(reprojected_bounds)
            gsd_x = (reprojected_bounds[2] - reprojected_bounds[0]) / width
            gsd_y = (reprojected_bounds[3] - reprojected_bounds[1]) / height
            gsd = (gsd_x + gsd_y) / 2
        else:
            bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
            gsd = src.res[0]
            
    print(f"Raster resolution (GSD): {gsd} meters")

    generate_raster_thumbnail(raster_path, raster_thumbnail, raster_style_file)
    style_info = parse_qml_classes_from_url(raster_style_file)

    nodata = None
    for cls in style_info:
        if cls.get("label", "").lower() in [x.lower() for x in constants.no_data_classnames_list]:
            nodata = cls.get("value")
            break
            
    if nodata is None:
        nodata = src.nodata if src.nodata is not None else 0

    print("No data class number = ",nodata)
    

    state_title = state.replace('_', ' ').title()
    block_title = block.replace('_', ' ').title()
    item_id = f"{os.path.splitext(raster_filename)[0]}"  
    
    item = pystac.Item(
        id=item_id,
        bbox=bbox,
        geometry=geom,
        datetime=start_date,
        properties={
            "title":f"{state_title}",
            # "description": f"Raster data for {os.path.splitext(raster_filename)[0]} in {block_title} of {state_title}",
            "description" : layer_description,
            "start_datetime": start_date.isoformat() + 'Z',
            "end_datetime": end_date.isoformat() + 'Z',
            "gsd": gsd,
        }
    )
    

    proj_ext = ProjectionExtension.ext(item, add_if_missing=True)
    proj_ext.epsg = proj_epsg
    proj_ext.bbox = bbox
    proj_ext.shape = [src.height, src.width]
        
    item.add_asset("data", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_path, start=base_dir)),
        media_type=MediaType.GEOTIFF,
        roles=["data"],
        title="Raster Layer"
    ))

    raster_ext = RasterExtension.ext(item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type, 
        spatial_resolution=gsd,
        nodata=nodata
    )
    raster_ext.bands = [raster_band]

    classification_ext = ClassificationExtension.ext(item.assets["data"], add_if_missing=True)
    stac_classes = []
    for cls in style_info:
        stac_class_obj = Classification.create(
            value=int(cls["value"]),
            name=cls.get("label") or f"Class {cls['value']}",
            description=cls.get("label"),
            color_hint=cls['color'].replace('#','')
        )
        stac_classes.append(stac_class_obj)
    classification_ext.classes = stac_classes

    item.add_asset("thumbnail", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_thumbnail, start=base_dir)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Raster Thumbnail"
    ))

    item.add_asset("style", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_style_file, start=base_dir)),
        media_type=MediaType.XML,
        roles=["metadata"],
        title="Raster Style (QML)"
    ))
    
    
   
    return item

### 8.Fetching the Vector items descriptions from QML and adding the assets

In [11]:
def parse_vector_descriptions(qml_path):
    tree = ET.parse(qml_path)
    root = tree.getroot()
    columns = []
    colnames = []
    coldesc = []
    for entry in root.findall(".//alias"):
        for attr_key, attr_value in entry.attrib.items():
            if (attr_key == 'field'):
                colnames.append(attr_value)
            if (attr_key == 'name'):
                coldesc.append(attr_value)
    vector_desc_df = pd.DataFrame([colnames,coldesc]).T
    vector_desc_df.columns = ['column_name','column_description']
    return vector_desc_df

### 9. Generating the vector items and adding to assets

In [12]:
def create_vector_item(state,district, block, vector_filename, vector_path,vector_desc_df, block_catalog_dir, vector_thumbnail, vector_style_file, base_dir, data_url,title,thumbnail_path,qml_path):
    start_date = constants.DEFAULT_START_DATE
    end_date = constants.DEFAULT_END_DATE

    response = requests.get(vector_path)
    response.raise_for_status()

    # if vector_filename.endswith('.geojson'):
    #     media_type = MediaType.GEOJSON

    media_type = MediaType.GEOJSON #TODO : temporary change. make it better
    
    gdf = gpd.read_file(response.text)
    gdf_wgs84 = gdf.to_crs(epsg=4326) if gdf.crs is None or gdf.crs.to_epsg() != 4326 else gdf

    bounds = gdf_wgs84.total_bounds
    bbox = [float(b) for b in bounds]
    geom = mapping(gdf_wgs84.union_all())
    
    generate_vector_thumbnail(vector_path, vector_thumbnail, qml_path)
    
    state_title = state.replace('_', ' ').title()
    block_title = block.replace('_', ' ').title()

    item_id = f"{os.path.splitext(vector_filename)[0]}"

    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bbox,
        datetime=start_date,
        properties={
            "title": f"{state_title}",
            "description": f"Vector data for {os.path.splitext(vector_filename)[0]} in {block_title} of {state_title}",
            "start_datetime": start_date.isoformat() + 'Z',
            "end_datetime": end_date.isoformat() + 'Z',
        }
    )
    
    column_desc_df = pd.read_csv('../data/column_descriptions.csv')
    column_desc_df.head()

    table_ext = TableExtension.ext(item, add_if_missing=True)
    vector_merged_df = gdf.dtypes.reset_index()
    vector_merged_df.columns = ['column_name','column_dtype']
    
    vector_desc_df = column_desc_df
    vector_desc_df = vector_desc_df[vector_desc_df['layer_name'] == 'admin_boundary']

    #vector_merged_df['column_description'] = '' #TODO: temporary
    #vector_merged_df = vector_merged_df.merge(vector_desc_df[['column_name',
                                                             #'column_name_description']],
                                                             # on='column_name', 
                                                             #  how='left') #.fillna('')
    vector_merged_df=vector_merged_df.merge(vector_desc_df[['column_name','column_name_description']],
                       on='column_name',
                       how='left').fillna('')
                                                          
    vector_merged_df.rename(columns={'column_name_description':'column_description'}, inplace=True)
    print("merged geojson with descriptions")
    
    print(vector_merged_df)
    print(item)
    #table_ext = TableExtension.ext(vector_merged_df, add_if_missing=True)
    table_ext.columns = [
        {
            "name": row['column_name'],
            "type": str(row['column_dtype']),
            "description" : row['column_description']
        }
        for ind,row in vector_merged_df.iterrows()
    ]
    print("creating table using tabular extension")
    item.add_asset("data", Asset(
        href=os.path.join(data_url, os.path.relpath(vector_path, start=base_dir)),
        media_type=media_type,
        roles=["data"],
        title="Vector Layer"
    ))
    print("added data asset")
    item.add_asset("thumbnail", Asset(
        href=os.path.join(data_url, os.path.relpath(vector_thumbnail, start=base_dir)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Vector Thumbnail"
    ))
    print("added thumbnail asset")
    item.add_asset("style", Asset(
        href=os.path.join(data_url, os.path.relpath(vector_style_file, start=base_dir)),
        media_type=MediaType.XML,
        roles=["metadata"],
        title="Vector Style"
    ))
    print("added style asset")
      
        
    return item

### 10. Generating the STAC for each block and iterating the flow for each layer

In [13]:
def generate_stac_for_block(info):
    base_dir = '../data/'
    corestack_dir = os.path.join(base_dir, 'CorestackCatalogs')
    
    
    state = info['state']
    district = info['district']
    block = info['block']
    
    block_title = block.replace('_', ' ').title()
    district_title=district.replace('_', ' ').title()
    state_title = state.replace('_', ' ').title()

    block_dir = os.path.join(corestack_dir, state, district, block)
    os.makedirs(block_dir, exist_ok=True)
    
    block_catalog = pystac.Catalog(
        id=block,
        title=f"{block_title}",
        description=f"STAC catalog for {block} block data in {district_title}, {state_title}"
    )

    layers_to_process = []
    if 'lulc_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'lulc_raster_file', 'style_key': 'lulc_raster_style_file', 'title':'LULC_raster'})
    if 'admin_boundary_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'admin_boundary_file', 'style_key': 'admin_boundary_style_file', 'title': 'admin_boundary'})
    if 'nrega_assets_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'nrega_assets_file', 'style_key': 'nrega_assets_style_file', 'title': 'nrega_assets'})
    if 'terrain_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'terrain_raster_file', 'style_key': 'terrain_raster_style_file', 'title': 'terrain_raster'})
    if 'terrain_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'terrain_vector_file', 'style_key': 'terrain_vector_style_file', 'title': 'terrain_vector'})
    if 'clart_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'clart_file', 'style_key': 'clart_style_file', 'title': 'CLART'})
    if 'surface_water_bodies_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'surface_water_bodies_file', 'style_key': 'surface_water_bodies_style_file', 'title': 'Surface_Water_bodies'})
    if 'drainage_lines_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'drainage_lines_file', 'style_key': 'drainage_lines_style_file', 'title': 'Drainage_lines'})             
    if 'change_detection_raster_Afforestation_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Afforestation_file', 'style_key': 'change_detection_raster_Afforestation_style_file', 'title': 'Change_detection_raster_Afforestation'})
    if 'cropping_intensity_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'cropping_intensity_file', 'style_key': 'cropping_intensity_style_file', 'title': 'Cropping_intensity'})
    if 'tree_health_ccd_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_ccd_raster_file', 'style_key': 'tree_health_ccd_raster_style_file', 'title': 'Tree_health_ccd_raster_2022'})
    if 'Prec_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'Prec_annual_file', 'style_key': 'Prec_annual_style_file', 'title': 'Prec_annual'})
    if 'tree_health_ch_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_ch_raster_file', 'style_key': 'tree_health_ch_raster_style_file', 'title': 'tree_health_ch_raster_2021'})
    if 'tree_health_overall_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_overall_raster_file', 'style_key': 'tree_health_overall_raster_style_file', 'title': 'tree_health_overall_raster'})  
    if 'aquifer_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'aquifer_vector_file', 'style_key': 'aquifer_vector_style_file', 'title': 'aquifer_vector'}) 
    if 'soge_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'soge_vector_file', 'style_key': 'soge_vector_style_file', 'title': 'soge_vector'})   
    if 'restoration_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'restoration_file', 'style_key': 'restoration_style_file', 'title': 'restoration'})       
    if 'change_detection_raster_CropIntensity_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_CropIntensity_file', 'style_key': 'change_detection_raster_CropIntensity_style_file', 'title': 'change_detection_raster_CropIntensity'})
    if 'change_detection_raster_Deforestation_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Deforestation_file', 'style_key': 'change_detection_raster_Deforestation_style_file', 'title': 'change_detection_raster_Deforestation'})
    if 'change_detection_raster_Degradation_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Degradation_file', 'style_key': 'change_detection_raster_Degradation_style_file', 'title': 'change_detection_raster_Degradation'})
    if 'change_detection_raster_Urbanization_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Urbanization_file', 'style_key': 'change_detection_raster_Urbanization_style_file', 'title': 'change_detection_raster_Urbanization'})
    if 'drought_frequency_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'drought_frequency_file', 'style_key': 'drought_frequency_style_file', 'title': 'drought_frequency'})
    if 'runoff_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'runoff_annual_file', 'style_key': 'runoff_annual_style_file', 'title': 'runoff_annual'})
    if 'well_depth_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'well_depth_annual_file', 'style_key': 'well_depth_annual_style_file', 'title': 'well_depth_annual'})   
    if 'deltaG_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'deltaG_annual_file', 'style_key': 'deltaG_annual_style_file', 'title': 'deltaG_annual'})
    if 'deltaG_fortnight_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'deltaG_fortnight_file', 'style_key': 'deltaG_fortnight_style_file', 'title': 'deltaG_fortnight'})    
    
    print(layers_to_process)       
    
    for layer in layers_to_process:
        try:
            file_path = os.path.join(base_dir, info[layer['file_key']])
            file_path_raster = 'https://geoserver.core-stack.org:8443/geoserver/LULC_level_1/wcs?service=WCS&version=2.0.1&request=GetCoverage&CoverageId=LULC_level_1:LULC_23_24_badlapur_level_1&format=geotiff'
            file_path_vector = 'https://geoserver.core-stack.org:8443/geoserver/panchayat_boundaries/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=panchayat_boundaries%3Ajaunpur_badlapur&outputFormat=application%2Fjson'
            style_path = os.path.join(base_dir, info[layer['style_key']])
            
            stac_output_dir = os.path.join(base_dir, 'STAC_output')
            os.makedirs(stac_output_dir, exist_ok=True)
            
            if os.path.exists(file_path):
                thumbnail_filename = f'{block}_{os.path.splitext(info[layer["file_key"]])[0]}_thumbnail.png'
                thumbnail_path = os.path.join(stac_output_dir, thumbnail_filename)
                
                title = layer.get('title')
                
                if layer['type'] == 'raster':
                    style_path = 'https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Land/LULC0_12class.qml'
                    item = create_raster_item(state, district, block, info[layer['file_key']], file_path_raster, block_dir, thumbnail_path, style_path, base_dir, constants.data_url, title, stac_output_dir, thumbnail_path)
                else:
                    style_path = 'https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Demographic/Administrative-Boundary-Style.qml'
                    # vector_desc_df = parse_vector_descriptions(style_path)
                    vector_desc_df = pd.read_csv('../data/column_descriptions.csv')
                    vector_desc_df = vector_desc_df[vector_desc_df['layer_name'] == 'admin_boundary']
                    item = create_vector_item(state, district, block, info[layer['file_key']], file_path_vector,vector_desc_df, block_dir, thumbnail_path, style_path, base_dir, constants.data_url, title, stac_output_dir, style_path)
                block_catalog.add_item(item)
    
        except Exception as e:
            print(f"Error processing layer '{layer.get('title', 'N/A')}' for block '{block}': {e}")
            continue
            
    block_catalog.normalize_and_save(block_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    
    print(f"STAC catalog created for block: {block} in {district}, {state}")

### 11. Generating the root catalog and checking if already location & block exist 

In [14]:
def generate_root_catalog(blocks_info, base_dir, corestack_dir):
    root_catalog_path = os.path.join(corestack_dir, "catalog.json")

    if os.path.exists(root_catalog_path):
        root_catalog = pystac.read_file(root_catalog_path)
        print("Loaded existing root catalog.")
    else:
        os.makedirs(corestack_dir, exist_ok=True)
        root_catalog = pystac.Catalog(
            id="corestack_STAC",
            title=constants.root_catalog_title,
            description=constants.root_catalog_description
        )
        root_catalog.set_self_href(root_catalog_path)
        print("Created new root catalog.")

    # existing_root_children_ids = {child.id for child in root_catalog.get_children()}
    # root_catalog_modified = False

    state_collections = {}
    district_catalogs = {}

    for info in blocks_info:
        state = info['state']
        district = info['district']
        block = info['block']
        
        state_title = state.replace('_', ' ').title()
        district_title = district.replace('_', ' ').title()
        
        generate_stac_for_block(info)


        if state not in state_collections:
            state_dir = os.path.join(corestack_dir, state)
            state_collection_path = os.path.join(state_dir, "collection.json")

            if os.path.exists(state_collection_path):
                state_collection = pystac.read_file(state_collection_path)
                print(f"Loaded existing state collection: {state}")
            else:
                os.makedirs(state_dir, exist_ok=True)
                state_collection = pystac.Collection(
                    id=state,
                    title=f"{state_title}",
                    description=f"STAC Collection for data of {state} state.",
                    extent=pystac.Extent(
                spatial=pystac.SpatialExtent([0, 0, 0, 0]),
            temporal=pystac.TemporalExtent([[constants.DEFAULT_START_DATE, constants.DEFAULT_END_DATE]])
        ),
        license="https://spdx.org/licenses/CC-BY-4.0.html",
        providers=[
            pystac.Provider(
                name="CoRE Stack",
                roles=[pystac.ProviderRole.PRODUCER, pystac.ProviderRole.PROCESSOR, pystac.ProviderRole.HOST, pystac.ProviderRole.LICENSOR ],
                url="https://core-stack.org/"
            )
        ],
        keywords=["social-ecological", "sustainability","CoRE stack", block, district, state],
    )
                state_collection.normalize_and_save(state_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
                print(f"Created new state collection: {state}")

            root_catalog.add_child(state_collection)
            print(f"State collection '{state}' linked in root catalog.")
            state_collections[state] = state_collection    
            
        if (state, district) not in district_catalogs:
            district_dir = os.path.join(corestack_dir, state, district)
            district_catalog_path = os.path.join(district_dir, "catalog.json")
            if os.path.exists(district_catalog_path):
                district_catalog = pystac.read_file(district_catalog_path)
                print(f"Loaded existing district catalog: {district}")
            else:
                os.makedirs(district_dir, exist_ok=True)
                district_catalog = pystac.Catalog(
                    id=district,
                    title=f"{district_title}",
                    description=f"STAC catalog for data of {district} district"
                )
                district_catalog.normalize_and_save(district_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
                print(f"Created new district catalog: {district}")
            
            state_collections[state].add_child(district_catalog)
            print(f"Added district catalog '{district}' to state collection '{state}'.")
            district_catalogs[(state, district)] = district_catalog

        block_catalog_dir = os.path.join(corestack_dir, state, district, block)
        block_catalog = pystac.read_file(os.path.join(block_catalog_dir, 'catalog.json'))

        existing_child_ids_in_district = {child.id for child in district_catalogs[(state, district)].get_children()}
        
        if block not in existing_child_ids_in_district:
            district_catalogs[(state, district)].add_child(block_catalog)
            print(f"Added block catalog '{block}' to district catalog '{district}'.")
        else:
            print(f"Block '{block}' already exists in district catalog '{district}'.")

    root_catalog.normalize_and_save(corestack_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    print("\nRoot catalog and all sub-catalogs are up-to-date.")

In [15]:
for block_info in blocks_info:
    print(f"Processing block: {block_info['block']}")
    generate_stac_for_block(block_info)
    
generate_root_catalog(blocks_info, base_dir="../data/", corestack_dir="../data/CorestackCatalogs")

Processing block: badlapur
[{'type': 'raster', 'file_key': 'lulc_raster_file', 'style_key': 'lulc_raster_style_file', 'title': 'LULC_raster'}, {'type': 'vector', 'file_key': 'admin_boundary_file', 'style_key': 'admin_boundary_style_file', 'title': 'admin_boundary'}]
jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif
2019-07-01 00:00:00
2020-06-30 00:00:00


/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Raster resolution (GSD): 9.589087846259165 meters
https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Land/LULC0_12class.qml


/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Unique values in raster data: [ 1.  2.  3.  4.  6.  7.  8.  9. 10. 11. 12. nan]


...

Unique values in raster data: [ 1.  2.  3.  4.  6.  7.  8.  9. 10. 11. 12. nan]
Parsed QML values: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12]
Parsed QML colors: ['#ff0000', '#74ccf4', '#1ca3ec', '#0f5e9c', '#38761d', '#a9a9a9', '#bad93e', '#f59d22', '#ff9371', '#b3561d', '#a9a9a9']


/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


No data class number =  0


/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_10722/1991962823.py:81: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  symbol_element = renderer_element.find('.//symbol') or symbols.get(renderer_element.get('symbol'))


Error parsing QML file https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Demographic/Administrative-Boundary-Style.qml: name 'qml_path' is not defined
Applying default style due to parsing error.
Thumbnail saved to: ../data/STAC_output/badlapur_admin_boundary_jaunpur_badlapur_thumbnail.png
merged geojson with descriptions
   column_name column_dtype                                 column_description
0           id       object                                                   
1     ADI_2001        int32                 Aggregate Development Index (2001)
2     ADI_2011        int32                 Aggregate Development Index (2011)
3     ADI_2019        int32                 Aggregate Development Index (2019)
4   ASSET_2001        int32                              household assets 2001
5   ASSET_2011        int32  Percentage of household in a village with diff...
6   ASSET_2019        int32                              household assets 2019
7      BF_2001       object 

In [16]:
column_desc_df = pd.read_csv('../data/column_descriptions.csv')

In [17]:
column_desc_df.head()

,Sno.,layer_name,dataset_name (from DB),column_name,column_name_description,comments
0,0,admin_boundary,Admin Boundary,ASSET_2001,household assets 2001,NaN
1,1,admin_boundary,Admin Boundary,TOT_M,total male population,NaN
2,2,admin_boundary,Admin Boundary,dist_cen,district census code,NaN
3,3,admin_boundary,Admin Boundary,TOT_F,total female population,NaN
4,4,admin_boundary,Admin Boundary,ADI_2019,Aggregate Development Index (2019),NaN


In [18]:
file_path_vector = 'https://geoserver.core-stack.org:8443/geoserver/panchayat_boundaries/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=panchayat_boundaries%3Ajaunpur_badlapur&outputFormat=application%2Fjson'

In [19]:
response = requests.get(file_path_vector)
response.raise_for_status()

In [20]:
gdf = gpd.read_file(response.text)

In [21]:
gdf.columns

Index(['id', 'ADI_2001', 'ADI_2011', 'ADI_2019', 'ASSET_2001', 'ASSET_2011',
       'ASSET_2019', 'BF_2001', 'BF_2011', 'BF_2019', 'FC_2001', 'FC_2011',
       'FC_2019', 'F_ILL', 'F_LIT', 'F_SC', 'F_ST', 'MSW_2001', 'MSW_2011',
       'MSW_2019', 'M_ILL', 'M_LIT', 'M_SC', 'M_ST', 'No_HH', 'P_ILL', 'P_LIT',
       'P_SC', 'P_ST', 'TOT_F', 'TOT_M', 'TOT_P', 'block_cen', 'dist_cen',
       'district', 'state', 'state_cen', 'tehsil', 'vill_ID', 'vill_name',
       'geometry'],
      dtype='object')

In [22]:
gdf.head()

,id,ADI_2001,ADI_2011,ADI_2019,ASSET_2001,ASSET_2011,ASSET_2019,BF_2001,BF_2011,BF_2019,...,TOT_P,block_cen,dist_cen,district,state,state_cen,tehsil,vill_ID,vill_name,geometry
0,jaunpur_badlapur.1,0,0,0,0,0,0,0,0,0,...,0,0,0,JAUNPUR,UTTAR PRADESH,0,BADLAPUR,0,0,"MULTIPOLYGON (((82.50749 25.92285, 82.50818 25..."
1,jaunpur_badlapur.2,0,0,0,0,0,0,0,0,0,...,0,0,0,JAUNPUR,UTTAR PRADESH,0,BADLAPUR,0,MANIYAREPUR,"MULTIPOLYGON (((82.51544 25.95478, 82.51514 25..."
2,jaunpur_badlapur.3,0,0,0,0,0,0,0,0,0,...,0,0,0,JAUNPUR,UTTAR PRADESH,0,BADLAPUR,0,GAJAPUR,"MULTIPOLYGON (((82.5335 25.91724, 82.53354 25...."
3,jaunpur_badlapur.4,0,0,0,0,0,0,0,0,0,...,0,0,0,JAUNPUR,UTTAR PRADESH,0,BADLAPUR,0,BASUPUR,"MULTIPOLYGON (((82.33392 25.86041, 82.3338 25...."
4,jaunpur_badlapur.5,0,0,0,0,0,0,0,0,0,...,0,0,0,JAUNPUR,UTTAR PRADESH,0,BADLAPUR,0,KATEHARI,"MULTIPOLYGON (((82.5753 25.87248, 82.57526 25...."


In [23]:
gdf.dtypes.reset_index()

,index,0
0,id,object
1,ADI_2001,int32
2,ADI_2011,int32
3,ADI_2019,int32
4,ASSET_2001,int32
5,ASSET_2011,int32
6,ASSET_2019,int32
7,BF_2001,object
8,BF_2011,int32
9,BF_2019,int32


In [24]:
vector_merged_df = gdf.dtypes.reset_index()
vector_merged_df.columns = ['column_name','column_dtype']

In [25]:
vector_merged_df

,column_name,column_dtype
0,id,object
1,ADI_2001,int32
2,ADI_2011,int32
3,ADI_2019,int32
4,ASSET_2001,int32
5,ASSET_2011,int32
6,ASSET_2019,int32
7,BF_2001,object
8,BF_2011,int32
9,BF_2019,int32


In [26]:
vector_desc_df = column_desc_df

In [27]:
vector_merged_df.shape

(41, 2)

In [28]:
vector_desc_df.shape

(734, 6)

In [29]:
vector_desc_df = vector_desc_df[vector_desc_df['layer_name'] == 'admin_boundary']

In [30]:
vector_desc_df.shape

(40, 6)

In [31]:
vector_merged_df.merge(vector_desc_df[['column_name','column_name_description']],
                       on='column_name',
                       how='left')

,column_name,column_dtype,column_name_description
0,id,object,NaN
1,ADI_2001,int32,Aggregate Development Index (2001)
2,ADI_2011,int32,Aggregate Development Index (2011)
3,ADI_2019,int32,Aggregate Development Index (2019)
4,ASSET_2001,int32,household assets 2001
5,ASSET_2011,int32,Percentage of household in a village with diff...
6,ASSET_2019,int32,household assets 2019
7,BF_2001,object,Percentage of households in a village with dif...
8,BF_2011,int32,Percentage of households in a village with dif...
9,BF_2019,int32,Percentage of households in a village with dif...
